# Project 1 — Data Cleaning & Preparation

## Phase 5: Correct and Standardize Data Formats

The purpose of this phase is to check and standardize the formats of dates, numeric fields, and text fields in the cleaned dataset.

The data will only be changed where a formatting issue is identified or where standardization is useful for consistency. Values that are already correctly formatted will be left unchanged.

In [1]:
import pandas as pd

In [2]:
df = pd.read_excel("../data/processed/cleaned_dataset.xlsx")

In [3]:
df.shape

(1200, 14)

In [4]:
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [5]:
clean_df = df.copy()

## Starting Format Check

Before making any changes, I will review the current data types of the fields that are expected to contain dates, numbers, and text.

In [6]:
clean_df.dtypes

OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object

### Starting Point

The Phase 4 dataset contains 1,200 records and 14 columns. The current data types will be reviewed before deciding whether any formatting changes are required.

## 5.1 Date Format

The `Date` column will be checked to confirm that it is stored consistently as a date field and that the values can be used reliably for further analysis.

In [7]:
clean_df["Date"].dtype

dtype('<M8[ns]')

In [8]:
clean_df["Date"].head(10)

0   2023-01-04
1   2024-08-23
2   2024-02-27
3   2023-10-15
4   2025-05-08
5   2023-10-23
6   2025-06-17
7   2023-05-12
8   2025-04-02
9   2023-11-21
Name: Date, dtype: datetime64[ns]

In [9]:
clean_df["Date"].isnull().sum()

np.int64(0)

In [10]:
clean_df["Date"].min(), clean_df["Date"].max()

(Timestamp('2023-01-01 00:00:00'), Timestamp('2025-06-30 00:00:00'))

In [11]:
clean_df["Date"].dt.strftime("%Y-%m-%d").head(10)

0    2023-01-04
1    2024-08-23
2    2024-02-27
3    2023-10-15
4    2025-05-08
5    2023-10-23
6    2025-06-17
7    2023-05-12
8    2025-04-02
9    2023-11-21
Name: Date, dtype: object

### Date Finding

The `Date` column is stored as a datetime type and contains no missing values. The dates will be kept in a consistent `YYYY-MM-DD` representation when displayed or exported.

## 5.2 Numeric Formats

The numeric fields will be checked to confirm that they use appropriate numeric data types and consistent values for further analysis.

In [12]:
clean_df[[
    "Quantity",
    "UnitPrice",
    "ItemsInCart",
    "TotalPrice"
]].dtypes

Quantity         int64
UnitPrice      float64
ItemsInCart      int64
TotalPrice     float64
dtype: object

In [13]:
clean_df[[
    "Quantity",
    "UnitPrice",
    "ItemsInCart",
    "TotalPrice"
]].head(10)

,Quantity,UnitPrice,ItemsInCart,TotalPrice
0,5,570.62,7,2853.10
1,2,151.35,3,302.70
2,5,550.68,8,2753.40
3,1,273.19,5,273.19
4,4,626.01,8,2504.04
5,2,245.86,4,491.72
6,1,664.42,6,664.42
7,5,149.55,9,747.75
8,2,134.28,2,268.56
9,4,509.38,6,2037.52


In [14]:
clean_df[[
    "Quantity",
    "UnitPrice",
    "ItemsInCart",
    "TotalPrice"
]].describe()

,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200.000000,1200.000000,1200.000000,1200.000000
mean,2.945833,356.412750,5.485000,1053.968300
std,1.407557,197.177146,2.281983,819.856558
min,1.000000,11.390000,1.000000,11.390000
25%,2.000000,186.062500,4.000000,410.520000
50%,3.000000,364.210000,5.000000,823.615000
75%,4.000000,521.570000,7.000000,1578.475000
max,5.000000,699.930000,10.000000,3456.400000


In [15]:
price_columns = ["UnitPrice", "TotalPrice"]

{
    column: clean_df[column].apply(
        lambda x: len(str(x).split(".")[1]) if "." in str(x) else 0
    ).value_counts().sort_index().to_dict()
    for column in price_columns
}

{'UnitPrice': {1: 114, 2: 1086}, 'TotalPrice': {1: 260, 2: 940}}

In [16]:
clean_df[[
    "Quantity",
    "UnitPrice",
    "ItemsInCart",
    "TotalPrice"
]].isnull().sum()

Quantity       0
UnitPrice      0
ItemsInCart    0
TotalPrice     0
dtype: int64

## Standardizing Price Fields

`UnitPrice` and `TotalPrice` represent monetary values, so they will be rounded to two decimal places to provide consistent monetary precision.

The integer fields `Quantity` and `ItemsInCart` will remain unchanged.

In [17]:
clean_df["UnitPrice"] = clean_df["UnitPrice"].round(2)
clean_df["TotalPrice"] = clean_df["TotalPrice"].round(2)

In [18]:
clean_df[["UnitPrice", "TotalPrice"]].head(10)

,UnitPrice,TotalPrice
0,570.62,2853.10
1,151.35,302.70
2,550.68,2753.40
3,273.19,273.19
4,626.01,2504.04
5,245.86,491.72
6,664.42,664.42
7,149.55,747.75
8,134.28,268.56
9,509.38,2037.52


In [19]:
clean_df[["UnitPrice", "TotalPrice"]].dtypes

UnitPrice     float64
TotalPrice    float64
dtype: object

In [20]:
(clean_df["TotalPrice"] == (clean_df["Quantity"] * clean_df["UnitPrice"]).round(2)).all()

np.True_

### Numeric Format Finding

The numeric fields were reviewed for their data types, missing values, and value ranges. `Quantity` and `ItemsInCart` were already stored as integers and required no changes. `UnitPrice` and `TotalPrice` were rounded to two decimal places to provide consistent monetary precision.

## 5.3 Text Formats

The text fields will be reviewed for missing values, blank values, and leading or trailing whitespace.

Text will only be changed if an actual formatting issue is identified.

In [21]:
text_columns = clean_df.select_dtypes(include="object").columns

text_columns

Index(['OrderID', 'CustomerID', 'Product', 'ShippingAddress', 'PaymentMethod',
       'OrderStatus', 'TrackingNumber', 'CouponCode', 'ReferralSource'],
      dtype='object')

In [22]:
clean_df[text_columns].isnull().sum()

OrderID            0
CustomerID         0
Product            0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
CouponCode         0
ReferralSource     0
dtype: int64

In [23]:
blank_text_values = {
    column: clean_df[column].astype(str).str.strip().eq("").sum()
    for column in text_columns
}

blank_text_values

{'OrderID': np.int64(0),
 'CustomerID': np.int64(0),
 'Product': np.int64(0),
 'ShippingAddress': np.int64(0),
 'PaymentMethod': np.int64(0),
 'OrderStatus': np.int64(0),
 'TrackingNumber': np.int64(0),
 'CouponCode': np.int64(0),
 'ReferralSource': np.int64(0)}

In [24]:
whitespace_issues = {
    column: (clean_df[column].astype(str) != clean_df[column].astype(str).str.strip()).sum()
    for column in text_columns
}

whitespace_issues

{'OrderID': np.int64(0),
 'CustomerID': np.int64(0),
 'Product': np.int64(0),
 'ShippingAddress': np.int64(0),
 'PaymentMethod': np.int64(0),
 'OrderStatus': np.int64(0),
 'TrackingNumber': np.int64(0),
 'CouponCode': np.int64(0),
 'ReferralSource': np.int64(0)}

### Text Format Finding

The text and identifier fields were reviewed for missing values, blank values, and leading or trailing whitespace.

No text formatting issues were identified, so no text transformations were required.

Existing capitalization and identifier formats were kept unchanged because the values were already consistent.

## Final Phase 5 Verification

The standardized dataset will be checked before saving the final cleaned version.

In [25]:
clean_df.shape

(1200, 14)

In [26]:
clean_df.dtypes

OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object

In [27]:
(
    clean_df["Date"].isnull().sum(),
    clean_df["Quantity"].isnull().sum(),
    clean_df["UnitPrice"].isnull().sum(),
    clean_df["ItemsInCart"].isnull().sum(),
    clean_df["TotalPrice"].isnull().sum()
)

(np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0))

In [28]:
clean_df.to_excel("../data/processed/cleaned_dataset.xlsx", index=False)

### Final Phase 5 Result

The standardized dataset contains 1,200 records and 14 columns.

- Date values were verified and required no transformation.
- Quantity and ItemsInCart remained integer fields.
- UnitPrice and TotalPrice were rounded to two decimal places for consistent monetary precision.
- Text fields contained no missing, blank, or whitespace issues.
- No records were removed during this phase.

The final cleaned dataset was saved to:

`data/processed/cleaned_dataset.xlsx`

In [29]:
final_df = pd.read_excel("../data/processed/cleaned_dataset.xlsx")

final_df.shape

(1200, 14)

In [30]:
final_df[["UnitPrice", "TotalPrice"]].head(10)

,UnitPrice,TotalPrice
0,570.62,2853.10
1,151.35,302.70
2,550.68,2753.40
3,273.19,273.19
4,626.01,2504.04
5,245.86,491.72
6,664.42,664.42
7,149.55,747.75
8,134.28,268.56
9,509.38,2037.52


In [31]:
(final_df["TotalPrice"] == (final_df["Quantity"] * final_df["UnitPrice"]).round(2)).all()

np.True_